In [ ]:
# ==============================================================================
# ECONOMETRIC LAB: CAUSALITY TESTS AND DYNAMICS (EN)
# VIX TUPINIQUIM III (TVP-VAR) vs. BRAZIL EPU (BAKER, BLOOM & DAVIS)
# ==============================================================================

import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

warnings.filterwarnings('ignore')

plt.style.use(
    'seaborn-v0_8-whitegrid'
    if 'seaborn-v0_8-whitegrid' in plt.style.available
    else 'default'
)

def format_spines(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_color('black')
    ax.spines['right'].set_color('black')

print('=' * 80)
print(
    '--- [ECONOMETRIC BENCHMARK]: CAUSALITY TESTS (VIX TUPINIQUIM III vs BRAZIL EPU) ---'
)
print('=' * 80)

# ==============================================================================
# 1. DATA LOADING AND ALIGNMENT
# ==============================================================================
print('\n[1/5] Loading historical series (Brazil EPU and VIX Tupiniquim III)...')

df_epu = pd.read_excel('epu_baker_bloom_davis_brazil.xlsx', sheet_name='Brazil EPU Index')
df_epu['Data'] = pd.to_datetime(
    df_epu['year'].astype(str) + '-' + df_epu['month'].astype(str) + '-01'
)
df_epu = df_epu.rename(columns={'Brazil News-Based EPU': 'EPU'})

try:
    df_vix = pd.read_excel('vix_tupiniquim_iii_series_historicas.xlsx')
except Exception:
    df_vix = pd.read_csv('vix_tupiniquim_iii_series_historicas.csv', sep=';')

df_vix['Data'] = pd.to_datetime(df_vix['Data'])

# ==============================================================================
# 2. STL DECOMPOSITION AND MERGE
# ==============================================================================
print('\n[2/5] Applying STL Filter (period=13) to Brazil EPU...')
stl_epu = STL(df_epu['EPU'], period=13, robust=True).fit()
df_epu['EPU_SA'] = stl_epu.trend + stl_epu.resid

target_vix_col = 'VIX_Tupiniquim_III_Media100'

df_analysis = (
    pd.merge(
        df_vix[['Data', target_vix_col]],
        df_epu[['Data', 'EPU_SA']],
        on='Data',
        how='inner',
    )
    .sort_values('Data')
    .reset_index(drop=True)
)
print(
    f'[SAMPLE ALIGNMENT]: Synchronized sample with {len(df_analysis)} monthly observations.'
)

# ==============================================================================
# 3. STATIONARITY VERIFICATION (ADF) & DIFFERENCING
# ==============================================================================
print('\n' + '=' * 80)
print('                    1. STATIONARITY TESTS (ADF)')
print('=' * 80)

df_analysis['dVIX_III'] = df_analysis[target_vix_col].diff()
df_granger = df_analysis.dropna().reset_index(drop=True)

p_vix_level = adfuller(df_analysis[target_vix_col])[1]
p_vix_diff = adfuller(df_granger['dVIX_III'])[1]
p_epu_level = adfuller(df_analysis['EPU_SA'])[1]

print(
    f"-> VIX Tupiniquim III (Level)         | ADF P-value: {p_vix_level:.4f} -> {'Stationary I(0)' if p_vix_level < 0.05 else 'Non-Stationary I(1)'}"
)
print(
    f"-> Delta VIX Tupiniquim III (Diff 1)  | ADF P-value: {p_vix_diff:.4f} -> {'Stationary I(0)' if p_vix_diff < 0.05 else 'Non-Stationary I(1)'}"
)
print(
    f"-> Deseasonalized EPU (Level)         | ADF P-value: {p_epu_level:.4f} -> {'Stationary I(0)' if p_epu_level < 0.05 else 'Non-Stationary I(1)'}"
)
print('=' * 80)

# ==============================================================================
# 4. STANDARD GRANGER CAUSALITY TEST (Delta VIX III vs EPU_SA)
# ==============================================================================
max_lags = 3
print('\n' + '=' * 80)
print(
    f'        2. STANDARD GRANGER CAUSALITY TEST [Delta VIX vs EPU] (Lags 1 to {max_lags})'
)
print('=' * 80)

# Direction 1: Delta VIX III -> EPU_SA
print('\n[DIRECTION 1]: VIX III Variation (Market) -> EPU (News/Press)')
gc_vix_to_epu = grangercausalitytests(
    df_granger[['EPU_SA', 'dVIX_III']], maxlag=max_lags, verbose=False
)
for lag in range(1, max_lags + 1):
    p_val = gc_vix_to_epu[lag][0]['ssr_ftest'][1]
    status = 'REJECT H0 (Granger-Causes)' if p_val < 0.05 else 'Do Not Reject H0 (No Causality)'
    print(f'-> Lag {lag}: P-value = {p_val:.4f} | {status}')

# Direction 2: EPU_SA -> Delta VIX III
print('\n[DIRECTION 2]: EPU (News/Press) -> VIX III Variation (Market)')
gc_epu_to_vix = grangercausalitytests(
    df_granger[['dVIX_III', 'EPU_SA']], maxlag=max_lags, verbose=False
)
for lag in range(1, max_lags + 1):
    p_val = gc_epu_to_vix[lag][0]['ssr_ftest'][1]
    status = 'REJECT H0 (Granger-Causes)' if p_val < 0.05 else 'Do Not Reject H0 (No Causality)'
    print(f'-> Lag {lag}: P-value = {p_val:.4f} | {status}')
print('=' * 80)

# ==============================================================================
# 5. TODA-YAMAMOTO (1995) PROCEDURE
# ==============================================================================
print('\n' + '=' * 80)
print('            3. TODA-YAMAMOTO ROBUST CAUSALITY TEST')
print('=' * 80)

d_max = 1
var_model = VAR(df_analysis[[target_vix_col, 'EPU_SA']])
lag_order = var_model.select_order(maxlags=6)
k = max(lag_order.bic, 1)

print(f'-> Optimal VAR order in levels (k via BIC): {k} lag(s)')
print(f'-> Maximum assumed integration order (d_max): {d_max}')
print(f'-> Augmented VAR estimated with (k + d_max) = {k + d_max} lags in levels.\n')

def run_toda_yamamoto(df_data, y_name, x_name, k_lags, d_integration):
    df_ty = pd.DataFrame(index=df_data.index)
    df_ty['const'] = 1.0

    for i in range(1, k_lags + d_integration + 1):
        df_ty[f'{y_name}_lag{i}'] = df_data[y_name].shift(i)

    for i in range(1, k_lags + d_integration + 1):
        df_ty[f'{x_name}_lag{i}'] = df_data[x_name].shift(i)

    df_ty['target'] = df_data[y_name]
    df_reg = df_ty.dropna()

    X_mat = df_reg.drop(columns=['target'])
    y_vec = df_reg['target']

    ols_model = sm.OLS(y_vec, X_mat).fit(cov_type='HC1')

    restrictions = [f'{x_name}_lag{i} = 0' for i in range(1, k_lags + 1)]
    wald_formula = ', '.join(restrictions)

    wald_test = ols_model.wald_test(wald_formula, scalar=True)
    return wald_test.statistic, wald_test.pvalue

stat_1, p_val_1 = run_toda_yamamoto(
    df_analysis, 'EPU_SA', target_vix_col, k, d_max
)
status_1 = 'REJECT H0 (Causes)' if p_val_1 < 0.05 else 'Do Not Reject H0 (No Causality)'
print('[DIRECTION 1 (TY)]: VIX Tupiniquim III -> EPU_SA (Levels)')
print(f'-> Wald Statistic: {stat_1:.4f} | P-value = {p_val_1:.4f} | {status_1}')

stat_2, p_val_2 = run_toda_yamamoto(
    df_analysis, target_vix_col, 'EPU_SA', k, d_max
)
status_2 = 'REJECT H0 (Causes)' if p_val_2 < 0.05 else 'Do Not Reject H0 (No Causality)'
print('\n[DIRECTION 2 (TY)]: EPU_SA (Levels) -> VIX Tupiniquim III')
print(f'-> Wald Statistic: {stat_2:.4f} | P-value = {p_val_2:.4f} | {status_2}')
print('=' * 80)

# ==============================================================================
# 6. COMPARATIVE CHART: VIX TUPINIQUIM III vs. BRAZIL EPU (DUAL AXIS)
# ==============================================================================
print('\nGenerating Comparative Chart: VIX Tupiniquim III vs. Brazil EPU (Dual Axis)...')

fig, ax1 = plt.subplots(figsize=(15, 5.5), dpi=300)

ax1.plot(
    df_analysis['Data'],
    df_analysis[target_vix_col],
    color='#1b5e20',
    linewidth=2.4,
    label='VIX Tupiniquim III (Base 100)',
)
ax1.set_xlabel('Year', fontweight='bold', fontsize=10)
ax1.set_ylabel(
    'VIX Tupiniquim III (Historical Mean = 100)',
    fontweight='bold',
    color='#1b5e20',
    fontsize=11,
)
ax1.tick_params(axis='y', labelcolor='#1b5e20')
ax1.axhline(100, color='#1b5e20', linestyle=':', linewidth=1.0, alpha=0.6)
ax1.grid(True, linestyle=':', alpha=0.4)

ax2 = ax1.twinx()
ax2.plot(
    df_analysis['Data'],
    df_analysis['EPU_SA'],
    color='#1f77b4',
    linewidth=1.9,
    linestyle='--',
    label='Brazil EPU (Deseasonalized)',
)
ax2.set_ylabel(
    'EPU Index (Baker, Bloom & Davis)',
    fontweight='bold',
    color='#1f77b4',
    fontsize=11,
)
ax2.tick_params(axis='y', labelcolor='#1f77b4')

format_spines(ax1)
format_spines(ax2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', frameon=False, fontsize=9.5)

plt.tight_layout()
plt.savefig('vix_iii_vs_epu_brasil_en.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n[SUCCESS]: Pipeline executed and 'vix_iii_vs_epu_brasil_en.png' successfully saved!")